# Prediction Visualizations

This notebook helps you visualize the model's predictions from `predictions.csv`.

## What you'll see:
1. Prediction accuracy metrics
2. Predicted vs actual yards distribution
3. Error analysis by predicted yards
4. Play-by-play prediction visualization
5. Confidence calibration plots

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Paths
PREDICTIONS_PATH = Path("predictions.csv")
VIZ_DIR = Path("visualizations")
VIZ_DIR.mkdir(exist_ok=True)

print("Loading predictions...")
df = pd.read_csv(PREDICTIONS_PATH)
print(f"Loaded {len(df)} predictions")

In [ ]:
# Calculate metrics by split
print("="*80)
print("PREDICTION METRICS BY DATASET SPLIT")
print("="*80)

for split in ['train', 'val', 'test']:
    split_df = df[df['dataset_split'] == split]
    if len(split_df) == 0:
        continue
    
    mae = np.abs(split_df['predicted_yards'] - split_df['actual_yards']).mean()
    rmse = np.sqrt(((split_df['predicted_yards'] - split_df['actual_yards'])**2).mean())

    print(f"\n{split.upper()} SET ({len(split_df)} predictions):")
    print(f"  MAE: {mae:.2f} yards")
    print(f"  RMSE: {rmse:.2f} yards")
    print(f"  Predicted - Mean: {split_df['predicted_yards'].mean():.2f}, "
          f"Median: {split_df['predicted_yards'].median():.2f}")
    print(f"  Actual - Mean: {split_df['actual_yards'].mean():.2f}, "
          f"Median: {split_df['actual_yards'].median():.2f}")

# Overall metrics
mae_all = np.abs(df['predicted_yards'] - df['actual_yards']).mean()
rmse_all = np.sqrt(((df['predicted_yards'] - df['actual_yards'])**2).mean())
print(f"\nOVERALL ({len(df)} predictions):")
print(f"  MAE: {mae_all:.2f} yards")
print(f"  RMSE: {rmse_all:.2f} yards")
print("="*80)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 16))

# Overall
axes[0, 0].scatter(df['actual_yards'], df['predicted_yards'], alpha=0.3, s=20)
axes[0, 0].plot([-10, 99], [-10, 99], 'r--', label='Perfect prediction')
axes[0, 0].set_xlabel('Actual Yards Gained')
axes[0, 0].set_ylabel('Predicted Yards Gained')
axes[0, 0].set_title('Overall - Predicted vs Actual Yards')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# By split
colors = {'train': 'blue', 'val': 'green', 'test': 'red'}
for idx, split in enumerate(['train', 'val', 'test']):
    ax = axes.flatten()[idx + 1]
    split_df = df[df['dataset_split'] == split]
    if len(split_df) == 0:
        ax.text(0.5, 0.5, f'No {split} data', ha='center', va='center', transform=ax.transAxes)
        continue
    
    ax.scatter(split_df['actual_yards'], split_df['predicted_yards'],
               alpha=0.3, s=20, c=colors[split])
    ax.plot([-10, 99], [-10, 99], 'r--', linewidth=2)
    ax.set_xlabel('Actual Yards Gained')
    ax.set_ylabel('Predicted Yards Gained')
    ax.set_title(f'{split.capitalize()} Set')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(VIZ_DIR / "predicted_vs_actual.png", dpi=300, bbox_inches='tight')
plt.show()
print("Saved: visualizations/predicted_vs_actual.png")

In [ ]:
df['error'] = df['predicted_yards'] - df['actual_yards']

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Error histogram
axes[0].hist(df['error'], bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(0, color='red', linestyle='--', linewidth=2, label='Zero error')
axes[0].set_xlabel('Prediction Error (yards)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Prediction Errors')
axes[0].legend()

# Error by actual yards
axes[1].scatter(df['actual_yards'], df['error'], alpha=0.3, s=20)
axes[1].axhline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Actual Yards Gained')
axes[1].set_ylabel('Prediction Error (yards)')
axes[1].set_title('Prediction Error vs Actual Yards')

plt.tight_layout()
plt.savefig(VIZ_DIR / "error_analysis.png", dpi=300, bbox_inches='tight')
plt.show()
print("Saved: visualizations/error_analysis.png")

In [ ]:
# Bin by confidence
df['confidence_bin'] = pd.cut(df['confidence'], bins=10, labels=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Confidence distribution
axes[0].hist(df['confidence'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Prediction Confidence')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Prediction Confidence')

# MAE by confidence bin
mae_by_conf = df.groupby('confidence_bin').apply(
    lambda x: np.abs(x['predicted_yards'] - x['actual_yards']).mean()
)
axes[1].plot(mae_by_conf.index, mae_by_conf.values, marker='o', linewidth=2)
axes[1].set_xlabel('Confidence Bin (0=low, 9=high)')
axes[1].set_ylabel('Mean Absolute Error (yards)')
axes[1].set_title('Prediction Error vs Confidence')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(VIZ_DIR / "confidence_analysis.png", dpi=300, bbox_inches='tight')
plt.show()
print("Saved: visualizations/confidence_analysis.png")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Predicted yards distribution
axes[0].hist(df['predicted_yards'], bins=30, alpha=0.5, label='Predicted', edgecolor='black')
axes[0].hist(df['actual_yards'], bins=30, alpha=0.5, label='Actual', edgecolor='black')
axes[0].set_xlabel('Yards Gained')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Predicted vs Actual Yards')
axes[0].legend()

# Box plot comparison
axes[1].boxplot([df['actual_yards'], df['predicted_yards']],
                labels=['Actual', 'Predicted'])
axes[1].set_ylabel('Yards Gained')
axes[1].set_title('Yards Distribution Comparison')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(VIZ_DIR / "distribution_comparison.png", dpi=300, bbox_inches='tight')
plt.show()
print("Saved: visualizations/distribution_comparison.png")

In [ ]:
# Show predictions for first 10 plays
sample_plays = df.groupby(['gameId', 'playId']).first().head(10)

print("\n" + "="*80)
print("SAMPLE PLAY PREDICTIONS")
print("="*80)
print(f"{'Game':<10} {'Play':<6} {'Predicted':<12} {'Actual':<10} {'Error':<10} {'Confidence':<12}")
print("-"*80)

for idx, row in sample_plays.iterrows():
    game_id, play_id = idx
    print(f"{game_id:<10} {play_id:<6} {row['predicted_yards']:>8.2f} yds  "
          f"{row['actual_yards']:>6.1f} yds  {row['error']:>7.2f} yds  {row['confidence']:>8.2%}")

## Custom Analysis

Use the cells below to create your own visualizations or analyze specific subsets of predictions.

Available columns: `gameId`, `playId`, `frameId`, `predicted_yards`, `predicted_class_yards`, `confidence`, `actual_yards`, `error`

In [ ]:
# Your custom analysis here
